# 01c · Features & Geometry `[local]`

**Goal:** find the *same* physical point in two different photos, then use those matches to work out how the photos relate geometrically. That's the basis of panoramas, AR, 3D reconstruction, visual SLAM and document alignment.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Keypoints as landmarks, descriptors as fingerprints, RANSAC as voting |
| 📐 Math | Harris corner score, Hamming distance, ratio test, homography, RANSAC |
| 🔨 From scratch | Harris corners, brute-force matcher, RANSAC line fit, homography from points (DLT) |
| 🚀 OpenCV | SIFT/ORB, matching, `findHomography`, template matching, camera calibration |
| 📊 Evaluate | Known-transform tests: SIFT vs. ORB under rotation and scale; calibration error |
| 🛠 Project | **Panorama stitcher** (by hand, then with `cv2.Stitcher`) |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from cvlib.data import opencv_sample, stitching_sample
from cvlib.viz import show

rng = np.random.default_rng(0)
box = cv2.imread(str(opencv_sample("box.png")))
scene = cv2.imread(str(opencv_sample("box_in_scene.png")))
show([box, scene], ["object", "scene: where is the object?"], size=4.5)

---
# 🧠 1 · Intuition

**Keypoints = landmarks.** If you had to find a friend's house from two photos taken from different spots, you wouldn't use a patch of plain wall. You'd use distinctive spots: a corner of a window, a sign. Those are **keypoints**: places that look unique and can be found again.
- Flat area: move a little window around and nothing changes, so it's useless.
- Edge: changes in one direction, but slides along the edge.
- **Corner:** changes whichever way you move, so it's pinned down. Corners make the best keypoints.

**Descriptors = fingerprints.** For each keypoint we summarize its neighborhood into a vector (128 numbers for SIFT, 256 bits for ORB). Same physical point → similar fingerprint, even after rotation, scaling or lighting changes.

**Matching = find the closest fingerprint.** For each keypoint in photo A, find the most similar descriptor in photo B.

**RANSAC = voting that ignores liars.** Some matches will be wrong. RANSAC repeatedly picks a tiny random set of matches, fits a transform, and counts how many other matches agree. The transform with the most supporters wins; disagreeing matches are thrown out as outliers.

**Homography = the "photo of a flat thing from another angle" transform** (the 3×3 perspective matrix from 01a). Two photos of a flat surface, or two photos taken from the same spot rotating the camera, are related by a homography. That's why panoramas work.

---
# 📐 2 · The math

### Harris corner score
At each pixel, look at the gradients $I_x, I_y$ in a small window and build the **structure tensor**:

$$M = \sum_{\text{window}} \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$

Its two eigenvalues say how much the window changes in its two main directions:
- both small → flat · one large → edge · **both large → corner**

Harris avoids computing eigenvalues with a shortcut:

$$R = \det(M) - k\,(\operatorname{trace} M)^2, \qquad k \approx 0.04$$

$R \gg 0$ → corner, $R < 0$ → edge, $|R|$ small → flat.

### Distances between descriptors
- **SIFT** (float vectors): Euclidean (L2) distance.
- **ORB** (binary strings): **Hamming distance** = number of differing bits (XOR, then count the 1s). Very fast.

### Lowe's ratio test
For each descriptor, find its **two** nearest neighbors at distances $d_1 \le d_2$. Keep the match only if

$$\frac{d_1}{d_2} < 0.75$$

If the best match is barely better than the second best, the keypoint is ambiguous (think repeated windows on a building), so drop it.

### Homography
A homography $H$ (3×3, defined up to scale → **8 unknowns**) maps point $(x, y)$ to $(x', y')$:

$$\begin{bmatrix} x'w \\ y'w \\ w \end{bmatrix} = H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} \quad\Rightarrow\quad x' = \frac{h_{11}x + h_{12}y + h_{13}}{h_{31}x + h_{32}y + h_{33}}, \;\; y' = \dots$$

Each point pair gives 2 equations, so **4 pairs** pin down 8 unknowns. With more pairs we solve by least squares (the **DLT** algorithm, via SVD).

### RANSAC
Repeat $N$ times: pick the minimum sample (2 points for a line, 4 for a homography) → fit → count **inliers** (points within a distance threshold). Keep the best. How many tries are needed?

$$N = \frac{\log(1 - p)}{\log(1 - w^s)}$$

$p$ = desired confidence (0.99), $w$ = inlier fraction, $s$ = sample size. With 50% inliers and $s=4$: $N \approx 72$. Cheap!

---
# 🔨 3 · Build it from scratch

### 3.1 · Harris corners

In [ ]:
def harris(gray, k=0.04, window_sigma=1.5):
    g = gray.astype(np.float32)
    Ix = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)            # gradients (you built Sobel in 01b)
    Iy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    # sum over a window = blur each product (Gaussian window = nearby pixels count more)
    Sxx = cv2.GaussianBlur(Ix * Ix, (0, 0), window_sigma)
    Syy = cv2.GaussianBlur(Iy * Iy, (0, 0), window_sigma)
    Sxy = cv2.GaussianBlur(Ix * Iy, (0, 0), window_sigma)
    det = Sxx * Syy - Sxy ** 2
    trace = Sxx + Syy
    return det - k * trace ** 2

def top_corners(R, n=150, min_dist=8):
    """Keep local maxima of R (non-max suppression), strongest n."""
    peaks = (R == cv2.dilate(R, np.ones((2 * min_dist + 1,) * 2, np.uint8))) & (R > 0.01 * R.max())
    ys, xs = np.nonzero(peaks)
    order = np.argsort(R[ys, xs])[::-1][:n]
    return np.stack([xs[order], ys[order]], 1)

gray_box = cv2.cvtColor(box, cv2.COLOR_BGR2GRAY)
R = harris(gray_box)
pts = top_corners(R)
vis = box.copy()
for x, y in pts:
    cv2.circle(vis, (int(x), int(y)), 3, (0, 0, 255), -1)

R_cv = cv2.cornerHarris(np.float32(gray_box), blockSize=3, ksize=3, k=0.04)
show([np.clip(R / R.max() * 255 * 20, 0, 255).astype(np.uint8), vis, np.clip(R_cv / R_cv.max() * 255 * 20, 0, 255).astype(np.uint8)],
     ["your Harris response R", f"top {len(pts)} corners", "cv2.cornerHarris (box window)"], cols=3, size=3.8)

Bright spots in R are corners. Your version and OpenCV's agree on *where* the corners are; their values differ a little because OpenCV sums over a box window instead of a Gaussian one.

### 3.2 · Brute-force matching with Hamming distance + ratio test

ORB descriptors are 32 bytes = 256 bits. We'll compute the full distance matrix ourselves.

In [ ]:
orb = cv2.ORB_create(nfeatures=1000)
kp1, des1 = orb.detectAndCompute(cv2.cvtColor(box, cv2.COLOR_BGR2GRAY), None)
kp2, des2 = orb.detectAndCompute(cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY), None)
print(f"object: {len(kp1)} keypoints | scene: {len(kp2)} keypoints | descriptor shape {des1.shape}, dtype {des1.dtype}")

def hamming_matrix(a, b):
    """Distance between every row of a and every row of b: XOR, then count the 1-bits."""
    a_bits = np.unpackbits(a, axis=1).astype(np.uint8)       # (N, 256) of 0/1
    b_bits = np.unpackbits(b, axis=1).astype(np.uint8)
    return (a_bits[:, None, :] != b_bits[None, :, :]).sum(-1) # (N, M)

def match_ratio(des_a, des_b, ratio=0.75):
    D = hamming_matrix(des_a, des_b)
    nn2 = np.argsort(D, axis=1)[:, :2]                        # best and second-best for each row
    d1 = D[np.arange(len(D)), nn2[:, 0]]
    d2 = D[np.arange(len(D)), nn2[:, 1]]
    keep = d1 < ratio * d2
    return [(i, nn2[i, 0], d1[i]) for i in np.nonzero(keep)[0]]

mine = match_ratio(des1, des2)
bf = cv2.BFMatcher(cv2.NORM_HAMMING)
cv_good = [m for m, n in bf.knnMatch(des1, des2, k=2) if m.distance < 0.75 * n.distance]
print(f"my matches: {len(mine)} | OpenCV BFMatcher + ratio test: {len(cv_good)}")

Same count, apart from tiny differences when two candidates are tied. The ratio test threw away most of the raw matches. Let's see how many survivors are actually right: draw them.

In [ ]:
vis_m = cv2.drawMatches(box, kp1, scene, kp2, sorted(cv_good, key=lambda m: m.distance)[:60], None,
                        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
show(vis_m, "ORB matches after the ratio test (some are still wrong)", size=10)

### 3.3 · RANSAC, from scratch, on a simple problem

Fit a line to points where 40% are garbage. Least squares gets dragged off; RANSAC ignores the outliers.

In [ ]:
n_in, n_out = 60, 40
x_in = rng.uniform(0, 10, n_in); y_in = 2 * x_in + 1 + rng.normal(0, 0.3, n_in)   # true line: y = 2x + 1
x_out = rng.uniform(0, 10, n_out); y_out = rng.uniform(-5, 30, n_out)
X = np.concatenate([x_in, x_out]); Y = np.concatenate([y_in, y_out])

def ransac_line(X, Y, n_iter=200, threshold=1.0):
    best_inliers = None
    for _ in range(n_iter):
        i, j = rng.choice(len(X), 2, replace=False)                  # minimal sample: 2 points
        if X[i] == X[j]:
            continue
        a = (Y[j] - Y[i]) / (X[j] - X[i]); b = Y[i] - a * X[i]
        inliers = np.abs(Y - (a * X + b)) < threshold               # who agrees?
        if best_inliers is None or inliers.sum() > best_inliers.sum():
            best_inliers = inliers
    a, b = np.polyfit(X[best_inliers], Y[best_inliers], 1)          # refit on all inliers
    return a, b, best_inliers

a_ls, b_ls = np.polyfit(X, Y, 1)
a_r, b_r, inl = ransac_line(X, Y)
print(f"truth        : y = 2.00x + 1.00")
print(f"least squares: y = {a_ls:.2f}x + {b_ls:.2f}   (pulled by outliers)")
print(f"RANSAC       : y = {a_r:.2f}x + {b_r:.2f}   ({inl.sum()} inliers found)")

xs = np.linspace(0, 10, 2)
plt.figure(figsize=(6, 4))
plt.scatter(X[inl], Y[inl], s=12, label="RANSAC inliers"); plt.scatter(X[~inl], Y[~inl], s=12, c="gray", label="outliers")
plt.plot(xs, a_ls * xs + b_ls, "r--", label="least squares"); plt.plot(xs, a_r * xs + b_r, "g", lw=2, label="RANSAC")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 3.4 · Homography from point pairs (DLT)

Each pair $(x, y) \to (x', y')$ gives two linear equations in the 9 entries of $H$. Stack them into a matrix $A$ and solve $A h = 0$. The solution is the singular vector with the smallest singular value.

In [ ]:
def homography_dlt(src, dst):
    rows = []
    for (x, y), (u, v) in zip(src, dst):
        rows.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        rows.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    _, _, Vt = np.linalg.svd(np.array(rows, np.float64))
    H = Vt[-1].reshape(3, 3)
    return H / H[2, 2]                                         # fix the scale so H[2,2] = 1

src = np.float64([[0, 0], [300, 0], [300, 200], [0, 200], [150, 100]])
H_true = np.array([[0.9, 0.2, 30], [-0.1, 1.1, 50], [0.0005, 0.0003, 1.0]])
dst_h = np.c_[src, np.ones(len(src))] @ H_true.T
dst = dst_h[:, :2] / dst_h[:, 2:]                              # divide by w: that's the perspective part

H_mine = homography_dlt(src, dst)
H_cv, _ = cv2.findHomography(src, dst, 0)                      # method 0 = plain least squares, no RANSAC
print("max |H_mine − H_true|:", np.abs(H_mine - H_true).max().round(8))
print("max |H_cv   − H_true|:", np.abs(H_cv - H_true).max().round(8))

---
# 🚀 4 · The OpenCV way

### 4.1 · SIFT vs. ORB

| | SIFT | ORB |
|---|---|---|
| Descriptor | 128 floats | 256 bits |
| Distance | L2 | Hamming |
| Scale/rotation invariant | yes (very good) | yes (decent) |
| Speed | slower | **very fast** (real-time on phones) |
| Patents | expired in 2020, now in main OpenCV | always free |

In [ ]:
gray_scene = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
sift = cv2.SIFT_create()
for name, det in [("SIFT", sift), ("ORB", cv2.ORB_create(nfeatures=1000))]:
    t0 = time.perf_counter(); kps, des = det.detectAndCompute(gray_scene, None); dt = (time.perf_counter() - t0) * 1000
    print(f"{name:4s}: {len(kps):4d} keypoints in {dt:6.1f} ms | descriptor {des.shape[1]} × {des.dtype}")

kps_s, _ = sift.detectAndCompute(gray_box, None)
vis_kp = cv2.drawKeypoints(box, kps_s[:150], None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
show(vis_kp, "SIFT keypoints: circle size = scale, line = orientation", size=6)

### 4.2 · Find the object: match → RANSAC homography → outline

In [ ]:
def locate(obj, scn, detector="sift", ratio=0.75, ransac_thresh=5.0):
    g1, g2 = cv2.cvtColor(obj, cv2.COLOR_BGR2GRAY), cv2.cvtColor(scn, cv2.COLOR_BGR2GRAY)
    if detector == "sift":
        det, norm = cv2.SIFT_create(), cv2.NORM_L2
    else:
        det, norm = cv2.ORB_create(nfeatures=2000), cv2.NORM_HAMMING
    k1, d1 = det.detectAndCompute(g1, None)
    k2, d2 = det.detectAndCompute(g2, None)
    if d1 is None or d2 is None or len(k1) < 2 or len(k2) < 2:
        return None, [], None, [], k1, k2
    good = [p[0] for p in cv2.BFMatcher(norm).knnMatch(d1, d2, k=2) if len(p) == 2 and p[0].distance < ratio * p[1].distance]
    if len(good) < 4:
        return None, good, None, [], k1, k2
    src = np.float32([k1[m.queryIdx].pt for m in good])
    dst = np.float32([k2[m.trainIdx].pt for m in good])
    H, inlier_mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_thresh)
    return H, good, inlier_mask, src, k1, k2

H, good, inlier_mask, _, k1, k2 = locate(box, scene, "sift")
h, w = box.shape[:2]
outline = cv2.perspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2), H)
vis_o = scene.copy()
cv2.polylines(vis_o, [np.int32(outline)], True, (0, 255, 0), 3)
inliers = [m for m, keep in zip(good, inlier_mask.ravel()) if keep]
print(f"{len(good)} ratio-test matches → {len(inliers)} RANSAC inliers ({len(inliers) / len(good):.0%})")
vis_in = cv2.drawMatches(box, k1, vis_o, k2, inliers, None, matchColor=(0, 255, 0), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
show(vis_in, "inlier matches only + the object's outline in the scene", size=10)

### 4.3 · Template matching, and why features exist

Template matching slides a small image over a big one and scores each position (it's correlation again, from 01b). Simple and fast, **but it only works when the target appears at the same size and rotation**.

In [ ]:
templ = box[40:180, 60:260]                                       # a crop of the box itself
methods = {"TM_CCOEFF_NORMED": cv2.TM_CCOEFF_NORMED, "TM_SQDIFF_NORMED": cv2.TM_SQDIFF_NORMED}

def best_match(img, templ, method):
    res = cv2.matchTemplate(img, templ, method)
    min_v, max_v, min_loc, max_loc = cv2.minMaxLoc(res)
    return (min_loc, 1 - min_v) if method == cv2.TM_SQDIFF_NORMED else (max_loc, max_v)

panels, titles = [], []
for label, target in [("same size", box), ("scaled 0.6× + rotated 25°", cv2.warpAffine(box, cv2.getRotationMatrix2D((w / 2, h / 2), 25, 0.6), (w, h), borderValue=(255, 255, 255)))]:
    (x, y), score = best_match(target, templ, cv2.TM_CCOEFF_NORMED)
    v = target.copy(); cv2.rectangle(v, (x, y), (x + templ.shape[1], y + templ.shape[0]), (0, 0, 255), 2)
    panels.append(v); titles.append(f"{label}: score {score:.2f}")
show([templ] + panels, ["template"] + titles, cols=3, size=3.8)

A score near 1.0 means a confident, correct match. Once the target is scaled and rotated, the score collapses and the box lands in the wrong place. Keypoint features (SIFT/ORB) were invented exactly for this.

### 4.4 · Camera calibration: measuring your lens

Real lenses bend straight lines (**distortion**, worst at the edges of wide-angle lenses). Calibration estimates:
- the **camera matrix** $K$: focal lengths $f_x, f_y$ and image center $c_x, c_y$
- **distortion coefficients** $k_1, k_2, p_1, p_2, k_3$

Recipe: photograph a chessboard from many angles → find its corners → solve for the parameters that best explain all the photos. The quality score is the **reprojection error**: how far (in pixels) the predicted corners land from the detected ones.

In [ ]:
pattern = (9, 6)                                              # inner corners per row, per column
square = 1.0                                                  # size of one square (any unit)
objp = np.zeros((pattern[0] * pattern[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:pattern[0], 0:pattern[1]].T.reshape(-1, 2) * square   # (0,0,0), (1,0,0), ... flat board, z=0

obj_points, img_points, used = [], [], []
names = [f"left{i:02d}.jpg" for i in [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14]]
for name in names:
    g = cv2.imread(str(opencv_sample(name)), cv2.IMREAD_GRAYSCALE)
    ok, c = cv2.findChessboardCorners(g, pattern)
    if ok:
        c = cv2.cornerSubPix(g, c.reshape(-1, 1, 2), (11, 11), (-1, -1),
                             (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))   # refine to sub-pixel accuracy
        obj_points.append(objp); img_points.append(c); used.append(name)
print(f"chessboard found in {len(used)}/{len(names)} images")

rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(obj_points, img_points, g.shape[::-1], None, None)
print(f"RMS reprojection error: {rms:.3f} px")
print("camera matrix K:\n", K.round(1))
print("distortion (k1, k2, p1, p2, k3):", dist.ravel().round(3))

In [ ]:
sample = cv2.imread(str(opencv_sample("left12.jpg")))
vis_c = sample.copy()
ok, c = cv2.findChessboardCorners(cv2.cvtColor(sample, cv2.COLOR_BGR2GRAY), pattern)
cv2.drawChessboardCorners(vis_c, pattern, c.reshape(-1, 1, 2), ok)
undistorted = cv2.undistort(sample, K, dist)
show([vis_c, sample, undistorted], ["detected corners", "original (edges bend)", "undistorted (lines straight)"], cols=3, size=4)

Look at the left edge of the board: curved in the original, straight after undistortion. Calibration is the first step of any system that **measures** things with a camera: robot vision, AR, 3D reconstruction, measuring objects in mm.

---
# 🛠 Project · Panorama stitcher

Two photos are related by a homography when they show **a flat surface** or were taken from **the same spot while rotating the camera**. Our test images are overlapping photos of a large paper map of Budapest: a flat surface, so a homography fits exactly. Recipe:

1. detect + match features between neighboring photos
2. RANSAC homography from image B to image A
3. warp B into A's frame on a big enough canvas
4. paste A, and blend where they overlap

In [ ]:
pano_imgs = [cv2.imread(str(stitching_sample(f"budapest{i}.jpg"))) for i in (1, 2, 3)]
pano_imgs = [cv2.resize(im, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA) for im in pano_imgs]   # half size: faster
show(pano_imgs, ["budapest1", "budapest2", "budapest3"], cols=3, size=4)

In [ ]:
def stitch_pair(left, right, ratio=0.75, min_inliers=30):
    """Warp `right` into `left`'s frame and blend. Returns the panorama and match stats."""
    H, good, mask, _, _, _ = locate(right, left, "sift", ratio=ratio)     # maps right → left
    n_inliers = 0 if mask is None else int(mask.sum())
    if H is None or n_inliers < min_inliers:                               # sanity check before warping!
        raise ValueError(f"only {n_inliers} inliers from {len(good)} matches: the photos probably don't overlap")
    hl, wl = left.shape[:2]; hr, wr = right.shape[:2]
    # where do right's corners land? → size of the canvas
    corners_r = cv2.perspectiveTransform(np.float32([[0, 0], [wr, 0], [wr, hr], [0, hr]]).reshape(-1, 1, 2), H).reshape(-1, 2)
    all_pts = np.vstack([corners_r, [[0, 0], [wl, 0], [wl, hl], [0, hl]]])
    x_min, y_min = np.floor(all_pts.min(0)).astype(int)
    x_max, y_max = np.ceil(all_pts.max(0)).astype(int)
    T = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], np.float64)     # shift so nothing is at negative coords
    size = (x_max - x_min, y_max - y_min)
    warped_r = cv2.warpPerspective(right, T @ H, size)
    warped_l = cv2.warpPerspective(left, T, size)
    # feather blend: weight each image by distance to its own border
    def weight(img):
        m = (img.sum(2) > 0).astype(np.uint8)
        return cv2.distanceTransform(m, cv2.DIST_L2, 3)
    wl_, wr_ = weight(warped_l), weight(warped_r)
    total = wl_ + wr_; total[total == 0] = 1
    pano = (warped_l * (wl_ / total)[..., None] + warped_r * (wr_ / total)[..., None]).astype(np.uint8)
    return pano, len(good), int(mask.sum())

pano12, n_good, n_inl = stitch_pair(pano_imgs[0], pano_imgs[1])
print(f"budapest1 + budapest2: {n_good} matches, {n_inl} inliers")
show(pano12, "your 2-image panorama (feather-blended)", size=10)

Now the production version. `cv2.Stitcher` adds what we skipped: it estimates **camera rotations** (better than chaining homographies), does **bundle adjustment** (refines all cameras together), **exposure compensation**, **seam finding**, and **multi-band blending**.

In [ ]:
t0 = time.perf_counter()
status, pano_cv = cv2.Stitcher_create(cv2.Stitcher_PANORAMA).stitch(pano_imgs)
print(f"status {status} (0 = OK) in {time.perf_counter() - t0:.1f}s")
if status == 0:
    show(pano_cv, "cv2.Stitcher: 3 images", size=10)

---
# 📊 5 · Evaluate

### 5.1 · Known-transform test: SIFT vs. ORB under rotation and scale

Warp the box image by a **known** rotation + scale, then check whether each detector recovers it. Score = mean corner error of the recovered outline (success if < 5 px).

In [ ]:
def warp_known(img, angle, scale, canvas=(1000, 1000)):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    M[:, 2] += (canvas[0] / 2 - w / 2, canvas[1] / 2 - h / 2)      # center it on a bigger canvas
    H_true = np.vstack([M, [0, 0, 1]])
    return cv2.warpAffine(img, M, canvas, borderValue=(255, 255, 255)), H_true

corners_obj = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
rows = []
for angle in [0, 45, 90]:
    for scale in [0.25, 0.3, 0.5, 1.0, 2.0, 3.0]:
        target, H_true = warp_known(box, angle, scale)
        true_c = cv2.perspectiveTransform(corners_obj, H_true)
        row = {"angle": angle, "scale": scale}
        for det in ("sift", "orb"):
            Hh, _, m, _, _, _ = locate(box, target, det)
            err = np.inf if Hh is None else np.linalg.norm(cv2.perspectiveTransform(corners_obj, Hh) - true_c, axis=2).mean()
            row[det] = err
        rows.append(row)

print(f"{'angle':>6}{'scale':>7}{'SIFT err':>11}{'ORB err':>11}")
for r in rows:
    fmt = lambda e: f"{e:9.1f}px" if e < 999 else "   failed"
    print(f"{r['angle']:>6}{r['scale']:>7}{fmt(r['sift']):>11}{fmt(r['orb']):>11}")
for det in ("sift", "orb"):
    ok = sum(r[det] < 5 for r in rows)
    print(f"{det.upper()} success rate: {ok}/{len(rows)}")

**Reading the table:**
- **Rotation** is no problem for either. Both estimate each keypoint's orientation.
- **Shrinking** is where they differ. ORB fails once the object is ~30% of its original size: its image pyramid only covers a limited scale range, and tiny objects yield too few keypoints. SIFT keeps working.
- **Enlarging** works for both, but ORB's error grows to 3–6 px, right at our 5 px cut-off. SIFT stays around 1 px.
- **Precision:** even when both succeed, SIFT's corners are usually more accurate (sub-pixel vs. a few pixels).

ORB was ~5× faster in 4.1. Choose by your constraints: ORB for real-time on weak hardware when scale doesn't change much, SIFT when robustness and accuracy matter more.

### 5.2 · Calibration quality: reprojection error per image

An RMS below ~0.5 px is good. A single bad image (blurry, wrong corners) can spoil the calibration, so check per image and drop outliers.

In [ ]:
per_image = []
for op, ip, rv, tv in zip(obj_points, img_points, rvecs, tvecs):
    proj, _ = cv2.projectPoints(op, rv, tv, K, dist)
    per_image.append(np.sqrt(np.mean(np.sum((proj.reshape(-1, 2) - ip.reshape(-1, 2)) ** 2, 1))))
plt.figure(figsize=(8, 3))
plt.bar(used, per_image); plt.xticks(rotation=45); plt.ylabel("RMS error (px)"); plt.title("Reprojection error per calibration image")
plt.axhline(0.5, color="r", ls="--", lw=1); plt.tight_layout(); plt.show()

---
# 🧪 6 · Experiments: predict first, then run

**Experiment 1: ratio test threshold.** At ratio 0.6, 0.75 and 0.95, predict how the number of matches and the inlier percentage change.

In [ ]:
# Prediction: ...
for ratio in [0.6, 0.75, 0.95]:
    Hh, g_, m_, _, _, _ = locate(box, scene, "sift", ratio=ratio)
    print(f"ratio {ratio}: {len(g_):4d} matches, {int(m_.sum()):4d} inliers ({m_.sum() / len(g_):.0%})")

**Experiment 2: RANSAC threshold.** What happens to the inlier count with a 1 px vs. a 20 px threshold? Which is safer?

In [ ]:
# Prediction: ...
for th in [1.0, 5.0, 20.0]:
    _, g_, m_, _, _, _ = locate(box, scene, "sift", ransac_thresh=th)
    print(f"threshold {th:>4} px → {int(m_.sum())} inliers of {len(g_)}")
# Too strict rejects good matches with small localization errors.
# Too loose lets wrong matches vote, which can bend the homography.

**Experiment 3: photos that don't overlap.** Try stitching `budapest1` with `budapest6`, which show different parts of the city. Predict: how many matches survive the ratio test, and what would a homography fitted to them look like?

In [ ]:
# Prediction: ...
b6 = cv2.resize(cv2.imread(str(stitching_sample("budapest6.jpg"))), None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
show([pano_imgs[0], b6], ["budapest1", "budapest6"], size=4)
try:
    stitch_pair(pano_imgs[0], b6)
except ValueError as e:
    print("refused:", e)
# Without the min_inliers check, RANSAC would still return *some* homography from a handful of
# accidental matches. Warping with it produces garbage, or a canvas so huge it runs out of memory.
# Always sanity-check a fitted model before using it.

---
# 📝 7 · Recap

**Cheat sheet**
- **Keypoint** = distinctive, re-findable spot (corners are best). **Descriptor** = fingerprint of its neighborhood.
- **Harris**: structure tensor; $R = \det M - k(\operatorname{tr} M)^2$; both eigenvalues large → corner.
- **SIFT**: 128 floats, L2, very robust to scale. **ORB**: 256 bits, Hamming, very fast.
- **Ratio test** ($d_1 / d_2 < 0.75$) removes ambiguous matches. **RANSAC** removes the remaining wrong ones.
- **Homography**: 3×3, 8 degrees of freedom, ≥4 point pairs. Valid for flat scenes or pure camera rotation.
- **Template matching** only works at the same scale and rotation.
- **Calibration**: chessboard photos → camera matrix + distortion. Judge it by reprojection error (< 0.5 px is good).
- **Evaluate with known transforms**: warp by something you know, check the recovery.

**Interview-style questions**
1. Why are corners better keypoints than edges or flat regions? Explain with the structure tensor.
2. SIFT vs. ORB: which for a real-time mobile AR app, which for offline 3D reconstruction, and why?
3. What problem does Lowe's ratio test solve? Give an example scene where it matters.
4. How many correspondences does a homography need? An affine transform? Why?
5. How many RANSAC iterations do you need with 30% inliers for a homography at 99% confidence?
6. When is a homography the *wrong* model for stitching two photos?
7. What does reprojection error measure in camera calibration?

**Next:** `01d_video_and_motion.ipynb`, where things start to move.